In [ ]:
# Cell 1 — imports
import re
import redivis
import pandas as pd

MIN_COMPANY_NAME_LEN = 4

In [ ]:
# Cell 2 — load our URL list from the v2 dataset (only need linkedin_url for SQL JOIN)
urls_table = redivis.user("ml2068").dataset("all_linkedin_urls_v2:ce7f").table("all_linkedin_urls_v2:gskd")
urls_df = urls_table.to_pandas_dataframe(variables=["linkedin_url"])

def _clean(u):
    if u is None or pd.isna(u):
        return None
    s = re.sub(r"^https?://(www\.)?", "", str(u).strip().rstrip("/"))
    return s if s.startswith("linkedin.com/in/") else None

urls_df["clean_linkedin_url"] = urls_df["linkedin_url"].apply(_clean)
urls_df = urls_df[urls_df["clean_linkedin_url"].notna()]
print(f"Our URLs: {len(urls_df):,} rows")
urls_df.head(3)

In [ ]:
# Cell 3 — join URLs → individual_user (server-side, only matched rows returned)
matched_users_df = redivis.query("""
    SELECT
        u.user_id, u.firstname, u.lastname, u.fullname,
        u.profile_linkedin_url, u.profile_title, u.numconnections,
        u.user_country, u.prestige,
        REGEXP_REPLACE(REGEXP_REPLACE(urls.linkedin_url, r'^https?://(www\\.)?', ''), r'/$', '') AS clean_linkedin_url
    FROM `all_linkedin_urls_v2:gskd` AS urls
    INNER JOIN `individual_user:xcsm` AS u
        ON REGEXP_REPLACE(REGEXP_REPLACE(urls.linkedin_url, r'^https?://(www\\.)?', ''), r'/$', '') = u.profile_linkedin_url
""").to_pandas_dataframe()

print(f"Matched users: {len(matched_users_df):,} rows")
matched_users_df.head(3)

In [ ]:
# Cell 4 — fetch positions via JOIN (avoids IN clause size limit)
positions_df = redivis.query("""
    SELECT
        p.user_id,
        p.company_cleaned,
        p.seniority,
        p.startdate,
        p.enddate
    FROM `individual_position:8xgp` AS p
    INNER JOIN `matched_users:kh5e` AS m
        ON p.user_id = m.user_id
""").to_pandas_dataframe()

print(f"Positions fetched: {len(positions_df):,} rows")
positions_df.head(3)

In [ ]:
# Cell 5 — build positions index {user_id: [company_cleaned, ...]}
positions_index = {}
for row in positions_df[["user_id", "company_cleaned"]].itertuples(index=False):
    uid = row.user_id
    if uid not in positions_index:
        positions_index[uid] = []
    positions_index[uid].append(row.company_cleaned)

print(f"Position index built for {len(positions_index):,} users")

In [ ]:
# Cell 6 — helper functions
import unicodedata
from difflib import SequenceMatcher

SUFFIXES = {
    "jr", "sr", "ii", "iii", "iv",
    "phd", "ph.d", "ph.d.", "md", "m.d", "m.d.",
    "jd", "j.d", "j.d.", "mba", "m.b.a", "cpa", "cfa", "esq",
    "ba", "ma", "ms", "bs", "mpa", "mph",
    "icd.d", "icd", "ret", "retired",
    "usaf", "usa", "usmc", "usn", "uscg", "cm",
}

LEGAL_SUFFIXES = re.compile(
    r"\b(inc|llc|corp|corporation|ltd|limited|gmbh|co|group|plc|sa|ag|bv|nv|lp|llp|holdings|holding|international|intl)\b",
    re.IGNORECASE
)


def to_ascii(s):
    """Normalize accents and special chars → plain ASCII lowercase."""
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9\s]", "", s.lower()).strip()


def clean_person_name(name):
    """Strip credentials/suffixes and return cleaned name tokens."""
    if pd.isna(name):
        return []
    name = str(name).split(",")[0].strip()
    tokens = name.lower().split()
    while tokens and tokens[-1].rstrip(".") in SUFFIXES:
        tokens.pop()
    return tokens


def name_matches(revelio_fullname, our_name, our_name_clean=None):
    """Four-stage cascade: last name → hyphen → first+initial → difflib."""
    if pd.isna(revelio_fullname):
        return False
    rev = to_ascii(revelio_fullname)

    for name in [our_name_clean, our_name]:
        tokens = clean_person_name(name)
        if not tokens:
            continue
        last = to_ascii(tokens[-1])
        first = to_ascii(tokens[0]) if len(tokens) > 1 else ""

        # Stage 1: last name substring
        if last and last in rev:
            return True

        # Stage 2: hyphenated last name — check each part
        if "-" in last:
            if any(part in rev for part in last.split("-") if part):
                return True

        # Stage 3: first name present + last initial present
        if first and last and first in rev and last[0] in rev:
            return True

        # Stage 4: difflib similarity on full name
        our_full = to_ascii(" ".join(tokens))
        if our_full and SequenceMatcher(None, our_full, rev).ratio() >= 0.85:
            return True

    return False


def strip_legal(name):
    """Strip legal suffixes and normalize to ASCII."""
    return to_ascii(LEGAL_SUFFIXES.sub("", str(name))).strip()


def company_in_positions(user_id, company_name, company_name_orig=None):
    """Bidirectional substring + token overlap after legal suffix stripping."""
    positions = positions_index.get(int(user_id), [])
    for name in [company_name, company_name_orig]:
        if pd.isna(name) or len(str(name)) < MIN_COMPANY_NAME_LEN:
            continue
        our = strip_legal(name)
        our_tokens = set(our.split()) - {""}
        if not our_tokens:
            continue
        for pos in positions:
            if pd.isna(pos):
                continue
            pos_clean = strip_legal(pos)
            # Bidirectional substring
            if our in pos_clean or pos_clean in our:
                return True
            # Token overlap: ≥50% of our tokens appear in position string
            pos_tokens = set(pos_clean.split())
            overlap = our_tokens & pos_tokens
            if len(overlap) / len(our_tokens) >= 0.5:
                return True
    return False


def clean_url(url):
    """Normalise to linkedin.com/in/<slug>."""
    if pd.isna(url):
        return None
    url = str(url).strip().rstrip("/")
    url = re.sub(r"^https?://(www\.)?", "", url)
    return url if url.startswith("linkedin.com/in/") else None

# ──────────────────────────────────────────────────────────
# Fuzzy company matcher — catches parent/subsidiary, abbreviations,
# and short names that the strict matcher misses. Computed in
# PARALLEL with the strict version so we can compare directly.
# ──────────────────────────────────────────────────────────

FUZZY_MIN_COMPANY_LEN = 3   # allow IBM, GE, HP, AT&T (vs strict's 4)
FUZZY_RATIO_THRESHOLD = 0.80


def company_in_positions_fuzzy(user_id, company_name, company_name_orig=None):
    """Strict match first; if that fails, try difflib similarity ≥ 0.80."""
    if company_in_positions(user_id, company_name, company_name_orig):
        return True

    positions = positions_index.get(int(user_id), [])
    for name in [company_name, company_name_orig]:
        if pd.isna(name) or len(str(name)) < FUZZY_MIN_COMPANY_LEN:
            continue
        our = strip_legal(name)
        if not our or len(our) < FUZZY_MIN_COMPANY_LEN:
            continue
        for pos in positions:
            if pd.isna(pos):
                continue
            pos_clean = strip_legal(pos)
            if not pos_clean or len(pos_clean) < FUZZY_MIN_COMPANY_LEN:
                continue
            if SequenceMatcher(None, our, pos_clean).ratio() >= FUZZY_RATIO_THRESHOLD:
                return True
    return False


In [ ]:
# Cell 7 — build lookup: clean_url → revelio user row
revelio_by_url = matched_users_df.drop_duplicates("clean_linkedin_url").set_index("clean_linkedin_url")

In [ ]:
# Cell 8 — load all_linkedin_urls v2 (with board_company + primary_company)
# v2 schema: separate board_company (WRDS) and primary_company (DEF 14A) so we can
# check Revelio work history against EITHER candidate per row.
all_urls_table = redivis.user("ml2068").dataset("all_linkedin_urls_v2:ce7f").table("all_linkedin_urls_v2:gskd")
all_people_df = all_urls_table.to_pandas_dataframe(
    variables=["person_name", "person_name_clean",
               "board_company", "primary_company", "company_name_clean",
               "source", "linkedin_url", "verified",
               "gvkey", "ticker", "search_anchor_used"]
)
print(f"all_linkedin_urls v2: {len(all_people_df):,} rows")
print(f"  By search_anchor_used:")
print(all_people_df["search_anchor_used"].value_counts().to_string())
print(f"  Has board_company:   {all_people_df['board_company'].notna().sum():,}")
print(f"  Has primary_company: {all_people_df['primary_company'].notna().sum():,}")
all_people_df.head(3)

In [ ]:
# Cell 9 — normalise URLs for join
all_people_df["clean_url"] = all_people_df["linkedin_url"].apply(clean_url)

In [ ]:
# Cell 10 — compute confirmation columns (two-tier: board, primary, either)
# For each URL row, check Revelio work history against board_company AND
# primary_company INDEPENDENTLY, then OR them into _either. This lets us:
#   Tier 1 (legacy / apples-to-apples): strong_match_board   = name AND board confirmed
#   Tier 2 (union / fairer for directors): strong_match_either = name AND (board OR primary)
# Also compute fuzzy variants in parallel.

revelio_url_match = []
revelio_name_confirmed = []
revelio_user_id_col = []

# Board-only
rev_co_conf_board = []
rev_co_conf_board_fuzzy = []
# Primary-only (False when primary_company is NaN)
rev_co_conf_primary = []
rev_co_conf_primary_fuzzy = []
# Union (OR of strict board + strict primary)
rev_co_conf_either = []
rev_co_conf_either_fuzzy = []

for row in all_people_df.itertuples(index=False):
    clean = getattr(row, "clean_url", None)
    rev = revelio_by_url.loc[clean] if (clean and clean in revelio_by_url.index) else None

    if rev is None:
        revelio_url_match.append(False)
        revelio_name_confirmed.append(False)
        revelio_user_id_col.append(None)
        rev_co_conf_board.append(False)
        rev_co_conf_board_fuzzy.append(False)
        rev_co_conf_primary.append(False)
        rev_co_conf_primary_fuzzy.append(False)
        rev_co_conf_either.append(False)
        rev_co_conf_either_fuzzy.append(False)
        continue

    revelio_url_match.append(True)
    uid = rev["user_id"]
    revelio_user_id_col.append(uid)
    revelio_name_confirmed.append(
        name_matches(rev["fullname"], row.person_name,
                     getattr(row, "person_name_clean", None))
    )

    board_co = getattr(row, "board_company", None)
    primary_co = getattr(row, "primary_company", None)

    # Strict
    b_strict = company_in_positions(uid, board_co, board_co)
    p_strict = company_in_positions(uid, primary_co, primary_co) if not pd.isna(primary_co) else False
    rev_co_conf_board.append(b_strict)
    rev_co_conf_primary.append(p_strict)
    rev_co_conf_either.append(b_strict or p_strict)

    # Fuzzy
    b_fuzzy = company_in_positions_fuzzy(uid, board_co, board_co)
    p_fuzzy = company_in_positions_fuzzy(uid, primary_co, primary_co) if not pd.isna(primary_co) else False
    rev_co_conf_board_fuzzy.append(b_fuzzy)
    rev_co_conf_primary_fuzzy.append(p_fuzzy)
    rev_co_conf_either_fuzzy.append(b_fuzzy or p_fuzzy)

all_people_df["revelio_url_match"] = revelio_url_match
all_people_df["revelio_name_confirmed"] = revelio_name_confirmed
all_people_df["revelio_user_id"] = revelio_user_id_col

all_people_df["revelio_company_confirmed_board"] = rev_co_conf_board
all_people_df["revelio_company_confirmed_primary"] = rev_co_conf_primary
all_people_df["revelio_company_confirmed_either"] = rev_co_conf_either

all_people_df["revelio_company_confirmed_board_fuzzy"] = rev_co_conf_board_fuzzy
all_people_df["revelio_company_confirmed_primary_fuzzy"] = rev_co_conf_primary_fuzzy
all_people_df["revelio_company_confirmed_either_fuzzy"] = rev_co_conf_either_fuzzy

# Backward compat: keep the legacy column name as alias for board (strict)
all_people_df["revelio_company_confirmed"] = rev_co_conf_board
all_people_df["revelio_company_confirmed_fuzzy"] = rev_co_conf_board_fuzzy

print("Done.")

In [ ]:
# Cell 11 — summary stats: Tier 1 (board), primary-only, Tier 2 (either), fuzzy variants
# Plus director-only breakdown stratified by search_anchor_used.

verified = all_people_df["verified"].fillna(False).astype(bool).tolist()

def count_strong(name_conf_list, co_conf_list, verified_list):
    return sum((n or v) and c for n, v, c in zip(name_conf_list, verified_list, co_conf_list))

total = len(all_people_df)
found = all_people_df["clean_url"].notna().sum()
matched = sum(revelio_url_match)
name_conf = sum(revelio_name_confirmed)

print(f"Total rows:                        {total:>8,}")
print(f"Has URL:                           {found:>8,}")
print(f"Revelio URL match:                 {matched:>8,}  ({matched/found*100:.1f}% of found)")
print(f"  Name confirmed:                  {name_conf:>8,}  ({name_conf/matched*100:.1f}%)")
print()
print(f"Company confirmation rate (of Revelio-matched):")
for label, col in [
    ("BOARD strict",   rev_co_conf_board),
    ("BOARD fuzzy",    rev_co_conf_board_fuzzy),
    ("PRIMARY strict", rev_co_conf_primary),
    ("PRIMARY fuzzy",  rev_co_conf_primary_fuzzy),
    ("EITHER strict",  rev_co_conf_either),
    ("EITHER fuzzy",   rev_co_conf_either_fuzzy),
]:
    n = sum(col)
    print(f"  {label:<16} {n:>8,}  ({n/matched*100:.1f}%)")
print()
print(f"Strong match rate (of Revelio-matched):")
for label, col in [
    ("Tier 1 (board strict)",   rev_co_conf_board),
    ("Tier 1 (board fuzzy)",    rev_co_conf_board_fuzzy),
    ("Tier 2 (either strict)",  rev_co_conf_either),
    ("Tier 2 (either fuzzy)",   rev_co_conf_either_fuzzy),
]:
    s = count_strong(revelio_name_confirmed, col, verified)
    print(f"  {label:<26} {s:>8,}  ({s/matched*100:.1f}%)")
print()

# ── Director-only breakdown stratified by search_anchor_used ──
print("=" * 70)
print("Director-only breakdown (source contains 'director' OR 'def14a')")
print("=" * 70)

source_str = all_people_df["source"].fillna("").astype(str)
is_director_row = source_str.str.contains("director|def14a", case=False, regex=True)

for anchor_label, anchor_mask in [
    ("ALL directors",     is_director_row),
    ("Original (board)",  is_director_row & (all_people_df["search_anchor_used"] == "board")),
    ("DEF 14A (primary)", is_director_row & (all_people_df["search_anchor_used"] == "primary")),
]:
    idx = anchor_mask[anchor_mask].index.tolist()
    if not idx:
        print(f"\n{anchor_label}: 0 rows")
        continue
    n_rows = len(idx)
    n_matched = sum(revelio_url_match[i] for i in idx)
    if n_matched == 0:
        print(f"\n{anchor_label}: {n_rows:,} rows, 0 Revelio-matched")
        continue
    s_board  = count_strong([revelio_name_confirmed[i] for i in idx],
                            [rev_co_conf_board[i] for i in idx],
                            [verified[i] for i in idx])
    s_either = count_strong([revelio_name_confirmed[i] for i in idx],
                            [rev_co_conf_either[i] for i in idx],
                            [verified[i] for i in idx])
    s_either_fuzzy = count_strong([revelio_name_confirmed[i] for i in idx],
                                  [rev_co_conf_either_fuzzy[i] for i in idx],
                                  [verified[i] for i in idx])
    print(f"\n{anchor_label}: {n_rows:,} rows, {n_matched:,} Revelio-matched")
    print(f"  Tier 1 strong (board strict):      {s_board:,}  ({s_board/n_matched*100:.1f}% of matched)")
    print(f"  Tier 2 strong (either strict):     {s_either:,}  ({s_either/n_matched*100:.1f}% of matched)")
    print(f"  Tier 2 strong (either fuzzy):      {s_either_fuzzy:,}  ({s_either_fuzzy/n_matched*100:.1f}% of matched)")
    print(f"  Δ Tier2 − Tier1 (strict):          {s_either - s_board:+,}")

In [ ]:
# Cell 12 — export summary with Tier 1 (board) and Tier 2 (either) strong-match columns
verified_s = all_people_df["verified"].fillna(False).astype(bool)

# Tier 1 — legacy, apples-to-apples (board company in work history)
all_people_df["strong_match_board"] = [
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified_s, rev_co_conf_board)
]
all_people_df["strong_match_board_fuzzy"] = [
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified_s, rev_co_conf_board_fuzzy)
]

# Primary-only (for decomposing where the lift comes from)
all_people_df["strong_match_primary"] = [
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified_s, rev_co_conf_primary)
]

# Tier 2 — union (board OR primary in work history) — the new fairer metric
all_people_df["strong_match_either"] = [
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified_s, rev_co_conf_either)
]
all_people_df["strong_match_either_fuzzy"] = [
    (rn or v) and rc
    for rn, v, rc in zip(revelio_name_confirmed, verified_s, rev_co_conf_either_fuzzy)
]

# Backward-compat: legacy strong_match == strong_match_board
all_people_df["strong_match"] = all_people_df["strong_match_board"]
all_people_df["strong_match_fuzzy"] = all_people_df["strong_match_board_fuzzy"]

output = all_people_df[[
    "linkedin_url",
    "person_name",
    "board_company",
    "primary_company",
    "company_name_clean",
    "source",
    "search_anchor_used",
    "ticker",
    "gvkey",
    "revelio_url_match",
    "revelio_name_confirmed",
    "revelio_company_confirmed_board",
    "revelio_company_confirmed_primary",
    "revelio_company_confirmed_either",
    "revelio_company_confirmed_board_fuzzy",
    "revelio_company_confirmed_primary_fuzzy",
    "revelio_company_confirmed_either_fuzzy",
    "revelio_user_id",
    "strong_match_board",          # Tier 1 (legacy, apples-to-apples)
    "strong_match_primary",        # decomposition
    "strong_match_either",         # Tier 2 (union — new headline)
    "strong_match_board_fuzzy",
    "strong_match_either_fuzzy",
    "strong_match",                # backward compat alias for strong_match_board
    "strong_match_fuzzy",          # backward compat alias for strong_match_board_fuzzy
]].copy()

output.to_csv("revelio_validation_summary_v2.csv", index=False)
print(f"Exported revelio_validation_summary_v2.csv ({len(output):,} rows)")
print(f"  Tier 1 strong (board strict):   {output['strong_match_board'].sum():,} "
      f"({output['strong_match_board'].mean()*100:.1f}%)")
print(f"  Tier 2 strong (either strict):  {output['strong_match_either'].sum():,} "
      f"({output['strong_match_either'].mean()*100:.1f}%)")
print(f"  Tier 2 strong (either fuzzy):   {output['strong_match_either_fuzzy'].sum():,} "
      f"({output['strong_match_either_fuzzy'].mean()*100:.1f}%)")
print(f"  Δ Tier2 − Tier1 (strict):       {(output['strong_match_either'].sum() - output['strong_match_board'].sum()):+,}")

redivis.current_notebook().create_output_table(output)
print("Output table created in Redivis workflow.")

In [ ]:
# Cell 12.5 — DIAGNOSTIC (print only, no export)
# For every row where strict EITHER company-match failed but fuzzy EITHER succeeded,
# replay the fuzzy match against BOTH board_company and primary_company to find
# which Revelio position string and ratio triggered the rescue.
# Use this to decide whether the fuzzy admits are signal or noise.

rescued = []  # list of (ratio, our_company, matched_position, source, which_anchor)

for i, (strict, fuzzy) in enumerate(zip(rev_co_conf_either,
                                        rev_co_conf_either_fuzzy)):
    if not (fuzzy and not strict):
        continue
    uid = revelio_user_id_col[i]
    if uid is None:
        continue
    row = all_people_df.iloc[i]
    board_co   = row.get("board_company")
    primary_co = row.get("primary_company")
    src        = row.get("source", "?")

    positions = positions_index.get(int(uid), [])
    best = (0.0, None, None)  # (ratio, position, anchor_label)
    for name_input, anchor_label in [(board_co, "board"), (primary_co, "primary")]:
        if name_input is None or pd.isna(name_input) or len(str(name_input)) < FUZZY_MIN_COMPANY_LEN:
            continue
        our = strip_legal(name_input)
        if not our or len(our) < FUZZY_MIN_COMPANY_LEN:
            continue
        for pos in positions:
            if pd.isna(pos):
                continue
            pos_clean = strip_legal(pos)
            if not pos_clean or len(pos_clean) < FUZZY_MIN_COMPANY_LEN:
                continue
            r = SequenceMatcher(None, our, pos_clean).ratio()
            if r >= FUZZY_RATIO_THRESHOLD and r > best[0]:
                best = (r, pos, f"{anchor_label}={name_input}")
    if best[1] is not None:
        rescued.append((best[0], best[2], best[1], src))

print(f"Total rescued by EITHER fuzzy (over EITHER strict): {len(rescued):,}")
print()
print("=" * 100)
print("TOP 30  (highest ratio — these should look like obvious variants)")
print("=" * 100)
print(f"  {'ratio':>5}  {'src':<14}  {'our company':<40}  →  Revelio position")
for r, our, pos, src in sorted(rescued, reverse=True)[:30]:
    print(f"  {r:>5.2f}  {str(src)[:14]:<14}  {str(our)[:40]:<40}  →  {pos}")

print()
print("=" * 100)
print("BOTTOM 30  (lowest ratio — most marginal, watch for false positives)")
print("=" * 100)
print(f"  {'ratio':>5}  {'src':<14}  {'our company':<40}  →  Revelio position")
for r, our, pos, src in sorted(rescued)[:30]:
    print(f"  {r:>5.2f}  {str(src)[:14]:<14}  {str(our)[:40]:<40}  →  {pos}")

print()
print("=" * 100)
print("RANDOM 20  (typical case)")
print("=" * 100)
import random
random.seed(42)
sample = random.sample(rescued, min(20, len(rescued)))
print(f"  {'ratio':>5}  {'src':<14}  {'our company':<40}  →  Revelio position")
for r, our, pos, src in sample:
    print(f"  {r:>5.2f}  {str(src)[:14]:<14}  {str(our)[:40]:<40}  →  {pos}")

In [ ]:
# Cell 13 — S&P 500 coverage analysis
# NOTE: v2 dataset has no `is_entity` column. Filter is skipped (treat all rows as people).
sp500_table = redivis.user("ml2068").dataset("sp500").table("sp500_companies")
sp500_df = sp500_table.to_pandas_dataframe(variables=["gvkey", "ticker", "company_name"])

def norm_gvkey(x):
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return None

sp500_gvkeys = set(norm_gvkey(g) for g in sp500_df["gvkey"].dropna())
sp500_gvkeys.discard(None)
print(f"S&P 500 companies: {len(sp500_gvkeys):,} unique gvkeys")

# Normalise gvkeys + tag S&P 500
all_people_df["gvkey_norm"] = all_people_df["gvkey"].apply(norm_gvkey)
all_people_df["is_sp500"] = all_people_df["gvkey_norm"].isin(sp500_gvkeys)

# Signal columns (already computed in Cell 10/12 — re-bind for clarity)
all_people_df["revelio_url_match_col"] = revelio_url_match
all_people_df["verified_bool"] = all_people_df["verified"].fillna(False).astype(bool)

# v2 dataset has no is_entity; treat all rows as people
if "is_entity" in all_people_df.columns:
    people_df = all_people_df[all_people_df["is_entity"] == False].copy()
else:
    people_df = all_people_df.copy()
print(f"People rows: {len(people_df):,}")

sp500_p = people_df[people_df["is_sp500"]]
non_sp500_p = people_df[~people_df["is_sp500"]]

# ── Helper ──────────────────────────────────────────────────────────────
def coverage_stats(df, label):
    total = len(df)
    has_url = df["linkedin_url"].notna().sum()
    verified = df["verified_bool"].sum()
    rev_match = df["revelio_url_match_col"].sum()
    s_board = df["strong_match_board"].sum()
    s_either = df["strong_match_either"].sum()
    print(f"{label} (n={total:,}):")
    print(f"  Has URL:            {has_url:>8,}  ({has_url/total*100:.1f}% of people)")
    print(f"  Verified (name):    {verified:>8,}  ({verified/total*100:.1f}% of people)")
    print(f"  Revelio matched:    {rev_match:>8,}  ({rev_match/has_url*100:.1f}% of URLs found)")
    if rev_match:
        print(f"  Tier 1 strong:      {s_board:>8,}  ({s_board/rev_match*100:.1f}% of Revelio matched)")
        print(f"  Tier 2 strong:      {s_either:>8,}  ({s_either/rev_match*100:.1f}% of Revelio matched)")
    print()

coverage_stats(sp500_p,     "S&P 500 companies")
coverage_stats(non_sp500_p, "Non-S&P 500 companies")
coverage_stats(people_df,   "All people")

# ── Source × anchor breakdown ─────────────────────────────────────────────
print("Strong match rate by source × search_anchor_used (Tier 1 / Tier 2):")
for src, grp in people_df.groupby("source"):
    for anchor, sub in grp.groupby("search_anchor_used"):
        rev = sub["revelio_url_match_col"].sum()
        s_b = sub["strong_match_board"].sum()
        s_e = sub["strong_match_either"].sum()
        rate_b = s_b / rev * 100 if rev else 0
        rate_e = s_e / rev * 100 if rev else 0
        print(f"  {str(src)[:30]:<30} anchor={anchor:<8} n={len(sub):>6,}  "
              f"T1={s_b:>5,} ({rate_b:.1f}%)  T2={s_e:>5,} ({rate_e:.1f}%)")
print()